In [ ]:
%run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/Initialization.ipynb"

In [ ]:
# Import packages
import os
import pickle

import numpy as np 
import pandas as pd 

import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

import tensorflow as tf
from tensorflow import keras

from scipy import stats
import scipy.optimize as optimize 

In [ ]:
# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Any results you write to the current directory are saved as output.

In [ ]:
org_data = pd.DataFrame()#pd.read_csv("/Users/Farid Srouji/Documents/Research/master/globaldata/options_VIX.csv")

from yahoo_fin.stock_info import get_data

VIX= get_data("^VIX", start_date="05/31/2020", end_date="26/08/2021", index_as_date = True, interval="1d")
SP500= get_data("^GSPC", start_date="05/31/2020", end_date="26/08/2021", index_as_date = True, interval="1d")

VIX["Date"]=VIX.index
VIX["Price"]=VIX.adjclose

SP500["Date"]=SP500.index
SP500["Price"]=SP500.adjclose

VIX = VIX.loc[:,['Date','Price']]
SP500 = SP500.loc[:,['Date','Price']]


In [ ]:
# Convert date data
VIX['Date'] = pd.to_datetime(VIX['Date'])
org_data["Date"] = VIX['Date']
org_data["vix"] = VIX["Price"]
org_data["sp500"] = SP500["Price"]

In [ ]:
org_data = org_data.ffill()

In [ ]:
pivot = ['Date','vix', 'sp500']

# Save the original data
full_data = org_data.copy()

# Shift the pivot data
full_data[pivot] = full_data[pivot].shift(periods=-1)

# Drop the last row
full_data = full_data.drop(full_data.index[-1])

In [ ]:
# Format foreign volatility indices as input data

def correspond_foreign_vol(date,data):
    """
    find the 'Price' of the date that is most recent to the given 'Date' in the 'data'
    """
    
    while True:
        date = date - pd.Timedelta('1 Day')  # go back one day 
        result_series = data['Price'].loc[data['Date']==date]  # find the 'Price' in the data that matches with the 'date'
        
        if not result_series.empty:  # if not empty (which means there is a row that matches with the 'date')
            result_series = result_series.reset_index()  # reset index
            result_value = result_series['Price'][0]  # and get the value
            return result_value

# Apply function

full_data['VIX'] = full_data['Date'].apply(correspond_foreign_vol,data=VIX)
# full_data['HSI'] = full_data['Date'].apply(correspond_foreign_vol,data=HSI)
# full_data['Nikkei'] = full_data['Date'].apply(correspond_foreign_vol,data=Nikkei)

In [ ]:
# Calculate return

SP500_yesterday = org_data['sp500'].shift(periods=1)  # Series made up of shifted VIX
return_array = (org_data['sp500']-SP500_yesterday) / SP500_yesterday  # Calculate return

In [ ]:
# GARCH model
def garch_forward(return_rate,variance,coefficients):
    ''' data type: float, float, 1d array(length=3)'''
    
    # Coefficients
    alpha,beta,omega = coefficients
    # Calculate
    return omega + alpha*return_rate*return_rate + beta*variance

In [ ]:
full_data['vix']

In [ ]:
# Set VIX as an initial variance in the model

initial_VIX = full_data['vix'][1] 
initial_variance = initial_VIX*initial_VIX/2520000 

In [ ]:
initial_VIX

In [ ]:
# Function for optimization

def garch_for_optimization(array): 
    ''' data type: 1d array(length=3)'''
    
    # Coeffcients
    alpha,beta,omega = array 
    
    # Variables
    sum_probability = 0 # to maximize
    variance = initial_variance
    
    for i in range(1,return_array_train.shape[0]):  # exclude the first value because it's nan.
        return_rate = return_array_train[i]  
        
        # in case something goes wrong
        if variance<=0:
            print("Negative variance")
            break
        
        # calculate probability in a single day
        probability = -np.log(variance) - return_rate * return_rate / variance
        
        # add to the sum
        sum_probability += probability
        
        # calculate next day's variance by GARCH
        variance = garch_forward(return_rate,variance,array)
   
    return -sum_probability # note the sign(-); because scipy.optimize requires a function to be minimized

In [ ]:
# Optimize
# if execute is set TRUE, optimization which takes some time will be initiated
execute = True    
return_array_train=return_array
if execute:
    bounds = optimize.Bounds([0,0,0],[1,1,np.inf])
    initial_guess = [0.14,0.76,2.97]  # V-Lab's estimate

    # Trust-constr performed the best 
    optimize_res_trust = optimize.minimize(garch_for_optimization,initial_guess,method='trust-constr',bounds=bounds)

    # # Pickle the result since it takes too much time
    with open('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/optimize_res_trust_VIX.pkl','wb') as file:
        pickle.dump(optimize_res_trust,file)

In [ ]:
optimize_res_trust

In [ ]:
# Load the pickle: optimize_res_trust
with open('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/optimize_res_trust.pkl','rb') as file:
    optimize_res_trust = pickle.load(file)

# Optimization result
print('optimization result:')
print('alpha =',optimize_res_trust.x[0])
print('beta =',optimize_res_trust.x[1])
print('omega =',optimize_res_trust.x[2])

In [ ]:
return_array

In [ ]:
optimize_res_trust.x

In [ ]:
# Estimate historical volatility with estimated coefficients

variance_array = np.zeros(full_data.shape[0],)  # Create array to store variance

# Values to be pre-assigned
variance_array[0] = np.nan  # Historical volatility cannot be estimated for the first one in full_data (06/03/2009)
variance_array[1] = initial_variance 

# Calculate historical volatilities using GARCH
for i in range(2,full_data.shape[0]):
    variance_array[i]=garch_forward(return_array[i-1],variance_array[i-1],optimize_res_trust.x)
    
# Adjust value to compare with VIX (elaborated in 2-3-2)  
historical = np.sqrt(variance_array * 252) * 100

# Add to the dataset
full_data['Historical Volatility'] = historical
full_data = full_data.dropna()  # Drop NA (because of nan in historical volatility)

In [ ]:
pd.DataFrame(data=variance_array)

In [ ]:
full_data.head(40)

In [ ]:
# Before visualizing, few matplotlib settings
mpl.rcParams['axes.labelsize'] = 'x-large'
mpl.rcParams['axes.labelpad'] = 5.5  # space between axis and label
mpl.rcParams['axes.titlesize'] = 'large'
mpl.rcParams['axes.titlepad'] = '20.0'

mpl.rcParams['legend.fontsize'] = 'x-large'

# mpl color settings
default_clrs = plt.rcParams['axes.prop_cycle'].by_key()['color']

# and seaborn settings
sns.set_style('darkgrid')

In [ ]:
# Compare with VIX (Visualization)
pd.plotting.register_matplotlib_converters()  # because of compatilbility issue with pd.Timestamp and matplotlib

fig, ax = plt.figure(figsize=(20,7)), plt.axes()
sns.lineplot(data=full_data,x='Date',y='vix',label='VIX (target)',ax=ax)
sns.lineplot(data=full_data,x='Date',y='Historical Volatility',label='Historical Volatility (prediction)',ax=ax)
plt.ylabel('Volatility')
plt.show()

In [ ]:
# Calculate error
abs_garch = abs(full_data['vix']-full_data['Historical Volatility'])
square_garch = (full_data['vix']-full_data['Historical Volatility'])**2

print('Total set')
print('MAE:',abs_garch.mean())
print('RMSE:',square_garch.mean()**0.5)

print('\nTest set ([-516:])')
print('MAE:',abs_garch[-516:].mean())
print('RMSE:',square_garch[-516:].mean()**0.5)

In [ ]:
full_data

In [ ]:
df_part3.reset_index(drop=True,inplace=True)
df_part3.drop(columns=["vix", "sp500"], inplace=True)

In [ ]:
full_data = pd.merge(full_data,df_part3,how='outer', on='Date')

In [ ]:
full_data = full_data.ffill()

In [ ]:
full_data = full_data.sort_values(by="Date")
full_data.index = full_data.Date

In [ ]:
# Split data into train data(including validation data) and test data
split_ratio = 0.7

data = full_data.set_index('Date')
len_data = full_data.shape[0]

train_org = full_data.iloc[:int(len_data*split_ratio),]
test_org = full_data.iloc[int(len_data*split_ratio):,]

print('train set',train_org.shape)
print('test set',test_org.shape)

# Data Preprocessing for Neural Net

 ## Normalize data

In [ ]:
# Normalize data

# Keep the original
train = train_org.copy()
test = test_org.copy()

# Variables that has to be normalized
to_normalize = ['vix','Historical Volatility'
                 ]
# Normalize
mean_train = train[to_normalize].mean()
std_train = train[to_normalize].std()

train[to_normalize] = (train[to_normalize]-mean_train)/std_train
test[to_normalize] = (test[to_normalize]-mean_train)/std_train

In [ ]:
# Select input variables
input_var = ['vix', 'Historical Volatility' ] 
print(f'total {len(input_var)} variables')

# Subsample data
x_train_org = train[input_var]
x_test_org = test[input_var]
y_train_org = train['vix']
y_test_org = test['vix']

print('x_train_org', x_train_org.shape)
print('y_train_org', y_train_org.shape)
print('x_test_org', x_test_org.shape)
print('y_test_org', y_test_org.shape)

In [ ]:
TIME_STEPS = 1  #predict with 5 days (+1 because we already shifted 1 day in 1-3)

# function
def reshape(data, time_steps = TIME_STEPS):
    stack = [data.iloc[i:i+time_steps].to_numpy() for i in range(len(data)-time_steps)]
    reshaped = np.stack(stack, axis=0) 
    return reshaped

# reshape
x_train_val = reshape(x_train_org) # train data and validation data
y_train_val = reshape(y_train_org)
x_test = reshape(x_test_org)
y_test = reshape(y_test_org)

print('x_train + x_val', x_train_val.shape)
print('y_train + y_val', y_train_val.shape)
print('x_test', x_test.shape)
print('y_test', y_test.shape)

In [ ]:
# Seperate validation data from train data
i = int(len(x_train_val) * 0.7)

x_train = x_train_val[:i]
x_val = x_train_val[i:]
y_train = y_train_val[:i]
y_val = y_train_val[i:]

print('train data:', x_train.shape[0])
print('validation data:', x_val.shape[0])
print('test data:', x_test.shape[0])

5. Neural Network
In this section, we build a neural network which includes historical volatility as an input variable.
Other option-related variables are also included in order to predict implied volatility.

We will use a couple of different neural network.

Vanilla NN
LSTM
GRU
5-1. Hyperparameter tuning
Bayesian optimization will be applied using kerastuner.

In [ ]:
from kerastuner import BayesianOptimization

# Function 'model_builder' builds and compiles neural network with given hyperparameters

# vanilla nn
def nn_builder(hp):
    # hp
    UNITS = hp.Int('UNITS', min_value = 16, max_value = 128, step = 16)
    ACTIVATION_1 = hp.Choice('ACTIVATION',values = ['relu','linear','tanh'])
    ACTIVATION_2 = hp.Choice('ACTIVATION',values = ['relu','linear','tanh'])
    
    # model instance
    model = keras.Sequential([
        keras.layers.Flatten(),
        keras.layers.Dense(units = UNITS, activation = ACTIVATION_1),
        keras.layers.Dropout(rate = 0.2),
        keras.layers.Dense(1, activation = ACTIVATION_2)
    ])
    
    # compile
    model.compile(optimizer = keras.optimizers.Adam(),
                  loss = 'mse', metrics = ['mae'])    
    
    return model

# lstm
def lstm_builder(hp):
    # hp
    UNITS_1 = hp.Int('UNITS_1', min_value = 16, max_value = 128, step = 16)
    UNITS_2 = hp.Int('UNITS_2', min_value = 8, max_value = 32, step = 8)
    ACTIVATION = hp.Choice('ACTIVATION',values = ['relu','linear','tanh'])
    
    # model instance
    model = keras.Sequential([
        # input_shape = (time_steps, # of features)
        keras.layers.LSTM(units = UNITS_1, input_shape = (None, x_train.shape[2]), return_sequences = False), 
        keras.layers.Dense(UNITS_2, activation = ACTIVATION),
        keras.layers.Dropout(rate = 0.2),
        keras.layers.Dense(1, activation = 'linear')
    ])
    
    # compile
    model.compile(optimizer = keras.optimizers.Adam(),
                  loss = 'mse', metrics = ['mae'])
    
    return model

# gru
def gru_builder(hp):
    # hp
    UNITS_1 = hp.Int('UNITS_1', min_value = 16, max_value = 128, step = 16)
    UNITS_2 = hp.Int('UNITS_2', min_value = 8, max_value = 32, step = 8)
    ACTIVATION = hp.Choice('ACTIVATION',values = ['relu','softmax','linear'])
    
    # model instance
    model = keras.Sequential([
        # input_shape = (time_steps, # of features)
        keras.layers.GRU(units = UNITS_1, input_shape = (None, x_train.shape[2]), return_sequences = False),
        keras.layers.Dense(UNITS_2, activation = ACTIVATION),
        keras.layers.Dropout(rate = 0.2),
        keras.layers.Dense(1, activation = 'linear')
    ])
    
    # compile
    model.compile(optimizer = keras.optimizers.Adam(),
                  loss = 'mse', metrics = ['mae'])
    
    return model

In [ ]:
# Implement earlystopping with keras callback 

PATIENCE = 5 # number of epochs with no improvement after which training will be stopped.

Earlystopping = keras.callbacks.EarlyStopping(monitor='val_loss',
                                              min_delta = 0.001,
                                              patience=PATIENCE, 
                                              mode='min', 
                                              restore_best_weights=True)

In [ ]:
# Search hyperparameters
SEED = 121

# NN
tuner_nn = BayesianOptimization(nn_builder,
                                objective = 'val_loss',
                                max_trials = 20,
                                seed = SEED,
                                directory = '/Users/Farid Srouji/Documents/kerastuner',
                                overwrite = True
                                )

tuner_nn.search(x_train, y_train, epochs=50, validation_data=(x_val, y_val), verbose=0, callbacks=[Earlystopping])

## Build model based on the optimized hyperparameters
besthp_nn = tuner_nn.get_best_hyperparameters()[0]
model_nn = tuner_nn.hypermodel.build(besthp_nn)


# lstm
tuner_lstm = BayesianOptimization(lstm_builder,
                            objective = 'val_loss',
                            max_trials = 20,
                            seed = SEED,
                            directory = '/Users/Farid Srouji/Documents/kerastuner')

tuner_lstm.search(x_train, y_train, epochs=50, validation_data=(x_val, y_val), verbose=0, callbacks=[Earlystopping])

## Build model based on the optimized hyperparameters
besthp_lstm = tuner_lstm.get_best_hyperparameters()[0]
model_lstm = tuner_lstm.hypermodel.build(besthp_lstm)


# gru
tuner_gru = BayesianOptimization(gru_builder,
                            objective = 'val_loss',
                            max_trials = 20,
                            seed = SEED,
                            directory = '/Users/Farid Srouji/Documents/kerastuner')

tuner_gru.search(x_train, y_train, epochs=50, validation_data=(x_val, y_val), verbose=0, callbacks=[Earlystopping])

## Build model based on the optimized hyperparameters
besthp_gru = tuner_gru.get_best_hyperparameters()[0]
model_gru = tuner_gru.hypermodel.build(besthp_gru)

In [ ]:
besthp_nn

In [ ]:
# Colors designated to each model
palette = default_clrs[:4]
palette.append(default_clrs[6])

In [ ]:
# Ensure reproducible result
print(SEED)
tf.random.set_seed(SEED)

In [ ]:
# Train model - NN
history = model_nn.fit(x_train, y_train, epochs=20, validation_data=(x_val, y_val), verbose=0, callbacks=[Earlystopping])

# Plot loss
plt.plot(history.history['loss'], label = 'loss')
plt.plot(history.history['val_loss'], label = 'val_loss', color=palette[2])
plt.legend()

# Loss
print('final loss:',history.history['loss'][-1])
print('final val_loss:',history.history['val_loss'][-1])

In [ ]:
model_nn.summary()

In [ ]:
# Train model - LSTM
history = model_lstm.fit(x_train, y_train, epochs=20, validation_data=(x_val, y_val), verbose=0, callbacks=[Earlystopping])

# Plot loss
plt.plot(history.history['loss'], label = 'loss')
plt.plot(history.history['val_loss'], label = 'val_loss',color=palette[3])
plt.legend()
# Loss
print('final loss:',history.history['loss'][-1])
print('final val_loss:',history.history['val_loss'][-1])

In [ ]:
model_lstm.summary()

In [ ]:
# Train model - GRU
history = model_gru.fit(x_train, y_train, epochs=20, validation_data=(x_val, y_val), verbose=0, callbacks=[Earlystopping])

# Plot loss
plt.plot(history.history['loss'], label = 'loss')
plt.plot(history.history['val_loss'], label = 'val_loss',color=palette[4])
plt.legend()

# Loss
print('final loss:',history.history['loss'][-1])
print('final val_loss:',history.history['val_loss'][-1])

In [ ]:
model_gru.summary()

# Result

In [ ]:
# Data for comparison
target_date = test_org['Date'].iloc[TIME_STEPS:] 
target_VIX = test_org['vix'].iloc[TIME_STEPS:]
predictions_hv = test_org['Historical Volatility'].iloc[TIME_STEPS:]

# Function to scale back predictions
def scale_back(predictions):
    predictions = [prediction[0] for prediction in predictions] # unpack array

    # scale back
    mean_vol = train_org['vix'].mean()
    std_vol = train_org['vix'].std() 
    predictions = [prediction * std_vol + mean_vol for prediction in predictions]
    
    return predictions

# Predict
predictions_nn = scale_back(model_nn.predict(x_test))
predictions_lstm = scale_back(model_lstm.predict(x_test))
predictions_gru = scale_back(model_gru.predict(x_test))

# pd.Series-ization
predictions_nn = pd.Series(predictions_nn, index=target_VIX.index, name='predictions_nn')
predictions_lstm = pd.Series(predictions_lstm, index=target_VIX.index, name='predictions_lstm')
predictions_gru = pd.Series(predictions_gru, index=target_VIX.index, name='predictions_gru')

In [ ]:
label_date = train_org['Date']#.iloc[TIME_STEPS:] 
label_VIX = train_org['vix']#.iloc[:TIME_STEPS]

In [ ]:
# Plot
fig1, ax1 = plt.figure(figsize=(20,7)), plt.axes()
fig2, ax2 = plt.figure(figsize=(20,7)), plt.axes()
fig3, ax3 = plt.figure(figsize=(20,7)), plt.axes()
fig4, ax4 = plt.figure(figsize=(20,7)), plt.axes()

# label_date 
ax1.plot(label_date, label_VIX, label='VIX (train data)')
ax2.plot(label_date, label_VIX, label='VIX (train data)')
ax3.plot(label_date, label_VIX, label='VIX (train data)')
ax4.plot(label_date, label_VIX, label='VIX (train data)')

# target data
ax1.plot(target_date, target_VIX, label='VIX (target data)')
ax2.plot(target_date, target_VIX, label='VIX (target data)')
ax3.plot(target_date, target_VIX, label='VIX (target data)')
ax4.plot(target_date, target_VIX, label='VIX (target data)')

# predictions
ax1.plot(target_date, predictions_hv, label='GARCH',color=palette[1])
ax2.plot(target_date, predictions_nn, label='GARCH-NN',color=palette[2])
ax3.plot(target_date, predictions_lstm, label='GARCH-LSTM',color=palette[3])
ax4.plot(target_date, predictions_gru, label='GARCH-GRU',color=palette[4])

# show
ax1.legend()
ax2.legend()
ax3.legend()
ax4.legend()
plt.show()

In [ ]:
# Plot
fig1, ax1 = plt.figure(figsize=(20,7)), plt.axes()
fig2, ax2 = plt.figure(figsize=(20,7)), plt.axes()
fig3, ax3 = plt.figure(figsize=(20,7)), plt.axes()
fig4, ax4 = plt.figure(figsize=(20,7)), plt.axes()

# target data
ax1.plot(target_date, target_VIX, label='VIX (target data)')
ax2.plot(target_date, target_VIX, label='VIX (target data)')
ax3.plot(target_date, target_VIX, label='VIX (target data)')
ax4.plot(target_date, target_VIX, label='VIX (target data)')

# predictions
ax1.plot(target_date, predictions_hv, label='GARCH',color=palette[1])
ax2.plot(target_date, predictions_nn, label='GARCH-NN',color=palette[2])
ax3.plot(target_date, predictions_lstm, label='GARCH-LSTM',color=palette[3])
ax4.plot(target_date, predictions_gru, label='GARCH-GRU',color=palette[4])

# show
ax1.legend()
ax2.legend()
ax3.legend()
ax4.legend()
plt.show()

# MSE and MAE

In [ ]:
# Calculate error
def cal_MSE(predictions):
    errors = (predictions - target_VIX)**2
    return errors.mean()

def cal_MAE(predictions):
    errors = abs(predictions - target_VIX)
    return errors.mean()


# Print error
print('GARCH')
print('MSE:',cal_MSE(predictions_hv))
print('MAE:',cal_MAE(predictions_hv))

print('GARCH-NN')
print('MSE:',cal_MSE(predictions_nn))
print('MAE:',cal_MAE(predictions_nn))

print('GARCH-LSTM')
print('MSE:',cal_MSE(predictions_lstm))
print('MAE:',cal_MAE(predictions_lstm))

print('GARCH-GRU')
print('MSE:',cal_MSE(predictions_gru))
print('MAE:',cal_MAE(predictions_gru))

# Number of best predictions

In [ ]:
# Compare day by day

# error list
errors_hv = abs(target_VIX - predictions_hv)
errors_nn = abs(target_VIX - predictions_nn)
errors_lstm = abs(target_VIX - predictions_lstm)
errors_gru = abs(target_VIX - predictions_gru)

# count the best models
best_models = []
for i in range(len(errors_hv)):
    each_predictions = [errors_hv.iloc[i], errors_nn.iloc[i], errors_lstm.iloc[i], errors_gru.iloc[i]]
    best_models.append(each_predictions.index(min(each_predictions)))

best_counts = [best_models.count(i) for i in range(4)]

    
# plot
fig, ax = plt.figure(figsize=(10,7)), plt.axes()
ax.pie(best_counts, labels=['GARCH','GARCH-NN','GARCH-LSTM','GARCH-GRU'], autopct='%.0f%%', startangle=90, colors=palette[1:])
plt.title('Number of days in which it performed the best')
plt.show()

# Number of predicting the right direction (up or down)

In [ ]:
# direction arrays
directions_VIX = target_VIX - target_VIX.shift()
directions_hv = predictions_hv - predictions_hv.shift()
directions_nn = predictions_nn - predictions_nn.shift()
directions_lstm = predictions_lstm - predictions_lstm.shift()
directions_gru = predictions_gru - predictions_gru.shift()

# multiplicate element-wise, and count the positivie elements
def count_pos(series):
    count = 0
    for x in series:
        if x>0:
            count += 1
    return count

direction_counts = [62,0,0,0,0]
direction_counts[1] = count_pos(directions_hv * directions_VIX)
direction_counts[2] = count_pos(directions_nn * directions_VIX)
direction_counts[3] = count_pos(directions_lstm * directions_VIX)
direction_counts[4] = count_pos(directions_gru * directions_VIX)

# plot
fig, ax = plt.figure(figsize=(10,7)), plt.axes()
ax.bar(['Total Days','GARCH','GARCH-NN','GARCH-LSTM','GARCH-GRU'],direction_counts, color=palette)
plt.title('Number of days in which it predicted the right direction(up or down)')
plt.show()

# direction prediction accuracy
print('GARCH')
print(direction_counts[1] / direction_counts[0]*100,'%')
print('GARCH-NN')
print(direction_counts[2] / direction_counts[0]*100,'%')
print('GARCH-LSTM')
print(direction_counts[3] / direction_counts[0]*100,'%')
print('GARCH-GRU')
print(direction_counts[4] / direction_counts[0]*100,'%')